### Import Libraries and Functions

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [13]:
output_dir = Path("../outputs")
raw_dir = output_dir / "raw"
summary_dir = output_dir / "summary_tables"

summary_dir.mkdir(parents=True, exist_ok=True)

In [15]:
scenario = "weighted_equal"

experiment_names = [
    "stock_proportional_weighted_equal",
    "boundary_shift_weighted_equal",
]

arrays_filenames = [
    f"sd_arrays_{experiment_name}.npz"
    for experiment_name in experiment_names
]

scenario_metadata_filenames = [
    f"scenario_metadata_{experiment_name}.csv"
    for experiment_name in experiment_names
]

### Load raw SD outputs

In [10]:
arrays_list = [
    np.load(raw_dir / filename, allow_pickle=True)
    for filename in arrays_filenames
]

scenario_metadata_list = [
    pd.read_csv(raw_dir / filename)
    for filename in scenario_metadata_filenames
]

for metadata in scenario_metadata_list:
    if "scenario_index" in metadata.columns:
        metadata.sort_values(
            "scenario_index",
            inplace=True,
        )
        metadata.reset_index(
            drop=True,
            inplace=True,
        )

    if (
        "stock_proportional_mean_time_days"
        not in metadata.columns
    ):
        metadata[
            "stock_proportional_mean_time_days"
        ] = np.nan

    if "boundary_shift_proportion" not in metadata.columns:
        metadata["boundary_shift_proportion"] = np.nan

t = arrays_list[0]["t"]
severity_levels = list(arrays_list[0]["severity_levels"])

for arrays in arrays_list[1:]:
    if not np.array_equal(t, arrays["t"]):
        raise ValueError(
            "The experiments use different time grids."
        )

    if severity_levels != list(arrays["severity_levels"]):
        raise ValueError(
            "The experiments use different severity levels."
        )

stocks = np.concatenate(
    [arrays["stocks"] for arrays in arrays_list],
    axis=0,
)

lambdas = np.concatenate(
    [arrays["lambdas"] for arrays in arrays_list],
    axis=0,
)

scenario_names = [
    scenario_name
    for arrays in arrays_list
    for scenario_name in arrays["scenario_names"]
]

scenario_metadata = pd.concat(
    scenario_metadata_list,
    ignore_index=True,
)

print(f"t shape: {t.shape}")
print(f"stocks shape: {stocks.shape}")
print(f"lambdas shape: {lambdas.shape}")
scenario_metadata.head()

t shape: (100001,)
stocks shape: (10, 100001, 3)
lambdas shape: (10, 100001, 3)


,scenario_index,scenario,deterioration_model,deterioration_label,stock_proportional_mean_time_days,boundary_shift_proportion,gatekeeping_policy,description
0,0,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,Stock proportional deterioration: six_months; ...
1,1,stock_proportional_two_years_weighted_equal,stock_proportional,two_years,730.0,NaN,weighted_equal,Stock proportional deterioration: two_years; g...
2,2,stock_proportional_five_years_weighted_equal,stock_proportional,five_years,1825.0,NaN,weighted_equal,Stock proportional deterioration: five_years; ...
3,3,stock_proportional_ten_years_weighted_equal,stock_proportional,ten_years,3650.0,NaN,weighted_equal,Stock proportional deterioration: ten_years; g...
4,0,boundary_shift_0_01_weighted_equal,boundary_shift,shift_0.01,NaN,0.01,weighted_equal,Boundary shift deterioration: 0.01 shift per s...


In [11]:
index_cols = [
    "scenario",
    "deterioration_model",
    "deterioration_label",
    "stock_proportional_mean_time_days",
    "boundary_shift_proportion",
    "gatekeeping_policy",
]

metadata_cols = [
    "scenario",
    "deterioration_model",
    "deterioration_label",
    "stock_proportional_mean_time_days",
    "boundary_shift_proportion",
    "gatekeeping_policy",
]


def summarise_array_by_scenario_and_severity(values, scenario_metadata, severity_levels):
    """Return standard summaries for an array with shape scenario × time × severity."""
    rows = []

    initial = values[:, 0, :]
    final = values[:, -1, :]
    minimum = values.min(axis=1)
    maximum = values.max(axis=1)
    mean = values.mean(axis=1)

    for scenario_idx, scenario_row in scenario_metadata.iterrows():
        for severity_idx, severity in enumerate(severity_levels):
            row = {col: scenario_row[col] for col in metadata_cols}
            row.update({
                "severity": severity,
                "initial": initial[scenario_idx, severity_idx],
                "final": final[scenario_idx, severity_idx],
                "minimum": minimum[scenario_idx, severity_idx],
                "maximum": maximum[scenario_idx, severity_idx],
                "mean": mean[scenario_idx, severity_idx],
            })
            rows.append(row)

    return pd.DataFrame(rows)

### Create summary table for stocks

In [14]:
stock_summary = summarise_array_by_scenario_and_severity(
    values=stocks,
    scenario_metadata=scenario_metadata,
    severity_levels=severity_levels,
)

stock_summary.to_csv(summary_dir / f"stock_summary_{scenario}.csv", index=False)
stock_summary.head()

,scenario,deterioration_model,deterioration_label,stock_proportional_mean_time_days,boundary_shift_proportion,gatekeeping_policy,severity,initial,final,minimum,maximum,mean
0,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,low,77406.716808,3687.757942,3687.757942,77406.716808,11212.333704
1,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,medium,46444.030085,3720.011089,3720.011089,51817.340052,15160.355573
2,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,high,30962.686723,160493.052552,30962.686723,160493.052552,136083.557907
3,stock_proportional_two_years_weighted_equal,stock_proportional,two_years,730.0,NaN,weighted_equal,low,77406.716808,19182.339920,19182.339920,77406.716808,37411.417958
4,stock_proportional_two_years_weighted_equal,stock_proportional,two_years,730.0,NaN,weighted_equal,medium,46444.030085,27396.305011,27396.305011,51600.134989,42042.498961


### Create referral summary table

In [17]:
referral_summary = summarise_array_by_scenario_and_severity(
    values=lambdas,
    scenario_metadata=scenario_metadata,
    severity_levels=severity_levels,
)

referral_summary = referral_summary.rename(columns={
    "mean": "mean_referrals",
    "maximum": "max_referrals",
})

referral_summary = referral_summary[[
    "scenario",
    "deterioration_model",
    "deterioration_label",
    "stock_proportional_mean_time_days",
    "boundary_shift_proportion",
    "gatekeeping_policy",
    "severity",
    "mean_referrals",
    "max_referrals",
]]

referral_summary.to_csv(summary_dir / f"referral_summary_{scenario}.csv", index=False)
referral_summary.head()

,scenario,deterioration_model,deterioration_label,stock_proportional_mean_time_days,boundary_shift_proportion,gatekeeping_policy,severity,mean_referrals,max_referrals
0,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,low,1.091548,7.535732
1,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,medium,1.475897,5.044544
2,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,high,13.248064,15.624388
3,stock_proportional_two_years_weighted_equal,stock_proportional,two_years,730.0,NaN,weighted_equal,low,3.642092,7.535732
4,stock_proportional_two_years_weighted_equal,stock_proportional,two_years,730.0,NaN,weighted_equal,medium,4.092939,5.023398


In [18]:
referral_summary["deterioration_model"].value_counts()

deterioration_model
boundary_shift        18
stock_proportional    12
Name: count, dtype: int64

### Create severity composition summary

In [19]:
stock_total = stocks.sum(axis=2)
lambda_total = lambdas.sum(axis=2)

stock_props = np.divide(
    stocks,
    stock_total[:, :, None],
    out=np.zeros_like(stocks),
    where=stock_total[:, :, None] > 0,
)

referral_props = np.divide(
    lambdas,
    lambda_total[:, :, None],
    out=np.zeros_like(lambdas),
    where=lambda_total[:, :, None] > 0,
)

final_composition = scenario_metadata[index_cols].copy()
final_composition["t"] = t[-1]

for severity_idx, severity in enumerate(severity_levels):
    final_composition[f"stock_prop_{severity}"] = stock_props[:, -1, severity_idx]
    final_composition[f"referral_prop_{severity}"] = referral_props[:, -1, severity_idx]

final_composition = final_composition[[
    "scenario",
    "deterioration_model",
    "deterioration_label",
    "stock_proportional_mean_time_days",
    "boundary_shift_proportion",
    "gatekeeping_policy",
    "t",
    "stock_prop_high",
    "stock_prop_medium",
    "stock_prop_low",
    "referral_prop_high",
    "referral_prop_medium",
    "referral_prop_low",
]]

final_composition.to_csv(summary_dir / f"final_composition_{scenario}.csv", index=False)
final_composition.head()

,scenario,deterioration_model,deterioration_label,stock_proportional_mean_time_days,boundary_shift_proportion,gatekeeping_policy,t,stock_prop_high,stock_prop_medium,stock_prop_low,referral_prop_high,referral_prop_medium,referral_prop_low
0,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,1825.0,0.955880,0.022156,0.021964,0.955880,0.022156,0.021964
1,stock_proportional_two_years_weighted_equal,stock_proportional,two_years,730.0,NaN,weighted_equal,1825.0,0.722582,0.163170,0.114248,0.722582,0.163170,0.114248
2,stock_proportional_five_years_weighted_equal,stock_proportional,five_years,1825.0,NaN,weighted_equal,1825.0,0.426864,0.287916,0.285220,0.426864,0.287916,0.285220
3,stock_proportional_ten_years_weighted_equal,stock_proportional,ten_years,3650.0,NaN,weighted_equal,1825.0,0.287932,0.299100,0.412968,0.287932,0.299100,0.412968
4,boundary_shift_0_01_weighted_equal,boundary_shift,shift_0.01,NaN,0.01,weighted_equal,1825.0,0.233105,0.244476,0.522418,0.233105,0.244476,0.522418


In [20]:
final_composition["deterioration_model"].value_counts()

deterioration_model
boundary_shift        6
stock_proportional    4
Name: count, dtype: int64

### Create one scenario-level table

In [ ]:
scenario_level_summary = scenario_metadata[index_cols].copy()

# Add final stock/referral composition information
scenario_level_summary = scenario_level_summary.merge(
    final_composition,
    on=index_cols,
    how="left",
)

scenario_level_summary.to_csv(
    summary_dir / f"scenario_level_summary_{scenario}.csv",
    index=False,
)

scenario_level_summary.head()

,scenario,deterioration_model,deterioration_label,stock_proportional_mean_time_days,boundary_shift_proportion,gatekeeping_policy,t,stock_prop_high,stock_prop_medium,stock_prop_low,referral_prop_high,referral_prop_medium,referral_prop_low
0,stock_proportional_six_months_weighted_equal,stock_proportional,six_months,180.0,NaN,weighted_equal,1825.0,0.955880,0.022156,0.021964,0.955880,0.022156,0.021964
1,stock_proportional_two_years_weighted_equal,stock_proportional,two_years,730.0,NaN,weighted_equal,1825.0,0.722582,0.163170,0.114248,0.722582,0.163170,0.114248
2,stock_proportional_five_years_weighted_equal,stock_proportional,five_years,1825.0,NaN,weighted_equal,1825.0,0.426864,0.287916,0.285220,0.426864,0.287916,0.285220
3,stock_proportional_ten_years_weighted_equal,stock_proportional,ten_years,3650.0,NaN,weighted_equal,1825.0,0.287932,0.299100,0.412968,0.287932,0.299100,0.412968
4,boundary_shift_0_01_weighted_equal,boundary_shift,shift_0.01,NaN,0.01,weighted_equal,1825.0,0.233105,0.244476,0.522418,0.233105,0.244476,0.522418
